## Objective & Tasks:
Data Processing: Clean and integrate these datasets. This should include, but not be limited to, handling missing values, duplicates, and possible outliers.

Data source: https://fueleconomy.gov/feg/ws/index.shtml

Step 1: Load bronze → Start processing raw data.

Step 2: Trim strings → Remove extra spaces for consistency.

Step 3: Convert timestamps → Make createdOn/modifiedOn proper timestamps for analysis.

Step 4: Check Nulls → Identify missing values.

Step 5: Check unique id → Ensure primary key uniqueness.

Step 6: Count nulls per column → Assess data quality and completeness.

Step 7: Quick profiling → Understand distributions, spot anomalies, negative or unexpected values.

Step 8: Write to Bronze Delta → Persist cleaned raw data.

Step 9: Register silver table → Make data accessible for SQL and BI tools.

Step 10: Display silver table → Quick visual check of processed data.

## Outcome:
The outcome of these steps is a cleaned, validated, and structured Silver layer table that is ready for analytics or further transformation

##Step 1: Load the bronze table to further process it in silver layer

In [0]:
df_silver = spark.read.table('epa_vehicle.bronze')
df_silver.printSchema()

##Stp 2: Trim string columns

Remove leading and trailing spaces from all string columns.

Prevents issues during type conversions (integer, decimal, date, boolean).

In [0]:
from pyspark.sql.functions import trim, col
from pyspark.sql.types import StringType

string_cols = [c.name for c in df_silver.schema.fields if isinstance(c.dataType, StringType)]
for c in string_cols:
    df_silver = df_silver.withColumn(c, trim(col(c)))

##Step 3: Convert createdOn and modifiedOn to timestamp

Current date format: Tue Jan 01 00:00:00 EST 2013

Why converting to timestamp:

- Correct sorting
- Correct filtering
- Date/time calculations
- Timezone correctness
- Better performance (predicate pushdown)


In [0]:
from pyspark.sql.functions import to_timestamp, col

#Convert createdOn and modifiedOn from string to timestamp.


df_silver = (
    df_silver
    .withColumn("createdOn",  to_timestamp(col("createdOn")))
    .withColumn("modifiedOn", to_timestamp(col("modifiedOn")))
)


#Validate the conversion
df_silver.select("createdOn", "modifiedOn").show(5, False)


##Step 4: Verify if there any Nulls

In [0]:
df_silver.filter(col("createdOn").isNull() | col("modifiedOn").isNull()).count()

##Step 5:Check the unique "id"

- According to https://fueleconomy.gov/feg/ws/index.shtml "id" returns a specific vehicle record, and it should be unique.

- Chek if the id has null values, and if it's duplicated


In [0]:
#Checking for null in id
df_silver.filter(col("id").isNull()).count()

In [0]:
from pyspark.sql.functions import col, count

#Checking for duplicate ids
duplicates = (
    df_silver.groupBy("id")
    .agg(count("*").alias("count"))
    .filter(col("count") > 1)
)

duplicates.show()


## Step 6: Counting the number of nulls in each column

It helps you see which columns have missing data.

In [0]:
from pyspark.sql.functions import col, sum

df_silver.select([sum(col(c).isNull().cast("int")).alias(c) for c in df_silver.columns]).display()


## Step 7: Quick data profiling step to understand the distribution and check for anomalies

In [0]:
df_silver.describe(["co2", "barrels08", "displ"]).show()

## Step 8: Write the DataFrame to Bronze Delta Table

In [0]:
# Define the silver path
silver_path = "dbfs:/mnt/epa/silver"

# Write df_silver to Delta format
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .save(silver_path)

print(f"Silver table saved at {silver_path}")

## Step 9: Register the silver table in the metastore for SQL access

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS epa_vehicle.silver
USING DELTA
LOCATION '{silver_path}'
""")

## Step 10: Display silver table

In [0]:
spark.read.table("hive_metastore.epa_vehicle.silver").display()